In [1]:
pip install requests beautifulsoup4 pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install schedule

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install streamlit pandas requests beautifulsoup4


Note: you may need to restart the kernel to use updated packages.


In [4]:
import streamlit as st
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

# ----------------- SCRAPERS -----------------
def scrape_cnn():
    urls = [
        "https://edition.cnn.com",
        "https://edition.cnn.com/world",
        "https://edition.cnn.com/politics",
        "https://edition.cnn.com/entertainment"
    ]
    results = []
    for url in urls:
        try:
            r = requests.get(url, timeout=10)
            soup = BeautifulSoup(r.content, 'html.parser')
            headlines = [h.get_text(strip=True) for h in soup.find_all("span", class_="container__headline-text")]
            results += [{"headline": h, "source": "CNN", "timestamp": datetime.now()} for h in headlines]
        except Exception as e:
            print(f"Error scraping CNN {url}: {e}")
    return results

def scrape_bbc():
    urls = [
        "https://www.bbc.com/news",
        "https://www.bbc.com/news/world",
        "https://www.bbc.com/news/technology"
    ]
    results = []
    for url in urls:
        try:
            r = requests.get(url, timeout=10)
            soup = BeautifulSoup(r.content, 'html.parser')
            headlines = [h.get_text(strip=True) for h in soup.find_all("span", class_="gs-c-promo-heading__title")]
            results += [{"headline": h, "source": "BBC", "timestamp": datetime.now()} for h in headlines]
        except Exception as e:
            print(f"Error scraping BBC {url}: {e}")
    return results

def scrape_buzzfeed():
    urls = [
        "https://www.buzzfeed.com",
        "https://www.buzzfeed.com/world",
        "https://www.buzzfeed.com/news"
    ]
    results = []
    for url in urls:
        try:
            r = requests.get(url, timeout=10)
            soup = BeautifulSoup(r.content, 'html.parser')
            headlines = [h.get_text(strip=True) for h in soup.find_all("h2") if "title" in "".join(h.get("class", []))]
            results += [{"headline": h, "source": "BuzzFeed", "timestamp": datetime.now()} for h in headlines]
        except Exception as e:
            print(f"Error scraping BuzzFeed {url}: {e}")
    return results

def live_scrape():
    all_data = scrape_cnn() + scrape_bbc() + scrape_buzzfeed()
    return pd.DataFrame(all_data)

# ----------------- STREAMLIT UI -----------------
st.title("📰 Clickbait vs Substance Detector (Live Headlines)")

if st.button("Scrape Headlines Now"):
    with st.spinner("Scraping headlines..."):
        df = live_scrape()
    if df.empty:
        st.warning("No headlines found.")
    else:
        st.success(f"Scraped {len(df)} headlines")
        st.dataframe(df)
        csv = df.to_csv(index=False).encode('utf-8')
        st.download_button("Download CSV", data=csv, file_name="live_headlines.csv", mime="text/csv")


2025-06-11 19:59:18.244 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-11 19:59:18.643 
  command:

    streamlit run C:\preet\envs\sentiment_env\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-06-11 19:59:18.644 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-11 19:59:18.644 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-11 19:59:18.645 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-11 19:59:18.646 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-11 19:59:18.647 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-11 19:59:18.648 Thread 'MainThread':

In [5]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from joblib import dump

# Load the dataset
df = pd.read_csv("C:/preet/clickbait_data.csv")  # Make sure this file is in the same directory

# Feature and target
X = df['headline']
y = df['clickbait']  # 1 = Clickbait, 0 = Substance

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train the model
model = LogisticRegression()
model.fit(X_train_vec, y_train)

# Evaluate
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

# Save model and vectorizer
dump(model, "clickbait_model.joblib")
dump(vectorizer, "vectorizer.joblib")
print("Model and vectorizer saved successfully.")


              precision    recall  f1-score   support

           0       0.96      0.98      0.97      3127
           1       0.98      0.96      0.97      3273

    accuracy                           0.97      6400
   macro avg       0.97      0.97      0.97      6400
weighted avg       0.97      0.97      0.97      6400

Model and vectorizer saved successfully.


In [7]:
import pandas as pd
import time

# Path to output CSV
output_path = "C:/Users/Admin/OneDrive/Desktop/scraping_metrics_live.csv"

# Write header once
columns = ["Website", "Success %", "Blocked URLs / Captchas %", "Error Rate by Type",
           "Throughput (pages/min)", "Data Loss Rate (%)", "Avg Response Time (sec)",
           "IP Block Freq (/hour)", "Pages Scraped"]

# Initialize CSV with headers (overwrite if exists)
pd.DataFrame(columns=columns).to_csv(output_path, index=False)

# Simulate scraping for 3 sites
sites = ["cnn.com", "buzzfeed.com", "bbc.com"]

for site in sites:
    # Simulate metrics (replace with actual scraped metrics)
    success = round(90 + 5 * (0.5 - time.time() % 1), 2)
    blocked = round(5 + 3 * (time.time() % 1), 2)
    errors = "404: 2%, Timeout: 1.5%"
    throughput = int(100 + 20 * (0.5 - time.time() % 1))
    data_loss = round(1 + 2 * (time.time() % 1), 2)
    response_time = round(0.8 + 0.3 * (time.time() % 1), 2)
    ip_block = int(1 + 3 * (time.time() % 1))
    pages = int(5000 + 500 * (time.time() % 1))

    row = {
        "Website": site,
        "Success %": success,
        "Blocked URLs / Captchas %": blocked,
        "Error Rate by Type": errors,
        "Throughput (pages/min)": throughput,
        "Data Loss Rate (%)": data_loss,
        "Avg Response Time (sec)": response_time,
        "IP Block Freq (/hour)": ip_block,
        "Pages Scraped": pages
    }

    # Append to CSV
    df = pd.DataFrame([row])
    df.to_csv(output_path, mode='a', header=False, index=False)

    print(f"Logged metrics for: {site}")
    time.sleep(1)  # Simulate delay between sites


Logged metrics for: cnn.com
Logged metrics for: buzzfeed.com
Logged metrics for: bbc.com


LLM-Prompts
for generating logs, I used chatgpt 
I faced errors in tf-idf

